# Melbourne Airbnb Data Cleaning and Exploration
RMIT University coursework by Hoang Nguyen.

# Task 1: Data Preparation


## Import and inspect data

In [ ]:
# Import library
import pandas as pd

import matplotlib.pyplot as plt

# Read the Airbnb data from the csv file
df = pd.read_csv("airbnb_input.csv")

# Check if we can read the file or not
df.head()

## Missing values

In [ ]:
# Check for missing values
df.isnull().sum()

### Fix the missing minimum nights

In [ ]:
# Check rows with missing minimum nights
df[df["minimum_nights"].isnull()]

In [ ]:
# Check the summary of minimum nights
df["minimum_nights"].describe()

In [ ]:
# Fill missing minimum nights with the median
df["minimum_nights"] = df["minimum_nights"].fillna(df["minimum_nights"].median())

In [ ]:
# Check missing minimum nights again
df["minimum_nights"].isnull().sum()

### Fix the missing number of reviews

In [ ]:
# Check rows with missing number of reviews
df[df["number_of_reviews"].isnull()]

In [ ]:
# Remove rows with missing number of reviews
df = df.dropna(subset=["number_of_reviews"])

In [ ]:
# Check missing number of reviews again
df["number_of_reviews"].isnull().sum()

In [ ]:
# Check for any missing values again
df.isnull().sum()

In [ ]:
# Fix data types for count columns
df["minimum_nights"] = df["minimum_nights"].astype(int)
df["number_of_reviews"] = df["number_of_reviews"].astype(int)

## Duplicated records

In [ ]:
# Check for duplicated listing IDs
df["id"].duplicated().sum()

In [ ]:
# Show all rows with duplicated listing IDs
df[df["id"].duplicated(keep=False)]

In [ ]:
# Remove duplicated listings
df = df.drop_duplicates(subset="id", keep="first")

In [ ]:
# Check duplicated listing IDs again
df["id"].duplicated().sum()

## Inconsistent and Invalid values

### Fix the inconsistent room types

In [ ]:
# Check the different room types
df["room_type"].value_counts()

In [ ]:
# Show rows with inconsistent room types
df[df["room_type"].isin(["Entire home/apartment", "Private rooms"])]

In [ ]:
# Fix inconsistent room types
df["room_type"] = df["room_type"].replace({
    "Entire home/apartment": "Entire home/apt",
    "Private rooms": "Private room"
})

In [ ]:
# Check the room types again
df["room_type"].value_counts()

In [ ]:
# Check the summary of numerical columns
df.describe()

### Fix invalid availability

In [ ]:
# Check invalid availability values
df[(df["availability_365"] < 0) | (df["availability_365"] > 365)]

In [ ]:
# Find the median from valid availability values
median_availability = df[(df["availability_365"] >= 0) & 
                        (df["availability_365"] <= 365)]["availability_365"].median()

median_availability

In [ ]:
# Replace invalid availability values with the median
df.loc[(df["availability_365"] < 0) | 
       (df["availability_365"] > 365), "availability_365"] = median_availability

In [ ]:
# Check invalid availability values again
df[(df["availability_365"] < 0) | (df["availability_365"] > 365)]

### Fix invalid latitude

In [ ]:
# Check positive latitude values
df[df["latitude"] > 0]

In [ ]:
# Fix positive latitude values to negative
df.loc[df["latitude"] > 0, "latitude"] = -df.loc[df["latitude"] > 0, "latitude"]

In [ ]:
# Check positive latitude values again
df[df["latitude"] > 0]

In [ ]:
# Final check of room types
df["room_type"].value_counts()

## Check for other data quality issues

In [ ]:
# Check neighbourhood values
df["neighbourhood"].value_counts()


In [ ]:
# Check for invalid date values
dates = pd.to_datetime(df["last_review"], format="%d/%m/%Y", errors="coerce")

df[dates.isna()]

In [ ]:
# Check if last 12 months reviews are more than total reviews
df[df["number_of_reviews_ltm"] > df["number_of_reviews"]]

In [ ]:
# Final check of the cleaned dataset
print("Dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicated IDs:", df["id"].duplicated().sum())

df.describe()

## Save the cleaned data

In [ ]:
# Save the cleaned dataset
df.to_csv("airbnb_cleaned.csv", index=False)

# Task 2: Data Exploration

In [ ]:
# Load the cleaned data for Task 2
df = pd.read_csv("airbnb_cleaned.csv")

## Task 2.1 

In [ ]:
# Find the median price for each room type
median_price = df.groupby("room_type")["price"].median()
median_price

In [ ]:
# Create boxplots of price by room type
df.boxplot(column="price", by="room_type", figsize=(10, 6))

plt.title("Price Distribution by Room Type")
plt.suptitle("")
plt.xlabel("Room Type")
plt.ylabel("Price ($)")
plt.xticks(rotation=20)
plt.show()

## Task 2.2 

In [ ]:
# Find the top five neighbourhoods with the highest mean price
mean_price_neighbourhood = df.groupby("neighbourhood")["price"].mean()
top_5_neighbourhoods = mean_price_neighbourhood.sort_values(ascending=False).head(5)
top_5_neighbourhoods

## Task 2.3 

In [ ]:
# Find the three neighbourhoods with the most Entire home/apt listings
entire_home = df[df["room_type"] == "Entire home/apt"]
top_3_neighbourhoods = entire_home["neighbourhood"].value_counts().head(3)
top_3_neighbourhoods

In [ ]:
# Compare prices using mean, median and standard deviation
price_comparison = entire_home[entire_home["neighbourhood"].isin(top_3_neighbourhoods.index)].groupby("neighbourhood")["price"].agg(["mean", "median", "std"])
price_comparison

## Task 2.4

In [ ]:
# Find the neighbourhood with the highest number of listings
top_neighbourhood = df["neighbourhood"].value_counts().head(1)
top_neighbourhood

In [ ]:
# Plot the geographical distribution of Airbnb listings
plt.figure(figsize=(10, 6))
plt.scatter(df["longitude"], df["latitude"], alpha=0.3, s=10)
plt.title("Geographical Distribution of Airbnb Listings")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

From the spatial plot, most Airbnb listings are concentrated around the central Melbourne area. Melbourne also has the highest number of listings with 3980. The listings become more spread out when moving further from the central area